In [1]:
import os
from getpass import getpass

if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [2]:
import weaviate
from weaviate.classes.init import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print(client.is_ready())

True


In [4]:
import weaviate.classes as wvc

COLLECTION_NAME = "Question"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

questions = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small"
    ),
    properties=[
        wvc.config.Property(
            name="question",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="answer",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Collection created:", COLLECTION_NAME)

Collection created: Question


In [5]:
uuid = questions.data.insert(
    {
        "question": "Leonardo da Vinci was born in this country.",
        "answer": "Italy",
        "category": "Culture",
    }
)

print(uuid)

e84322c0-4614-4180-8911-1d0f855064a3


In [6]:
obj = questions.query.fetch_object_by_id(uuid=uuid)

print(obj.properties)

{'answer': 'Italy', 'category': 'Culture', 'question': 'Leonardo da Vinci was born in this country.'}


In [7]:
obj = questions.query.fetch_object_by_id(
    uuid=uuid,
    include_vector=True,
)

print(obj.properties)
print("Vector keys:", obj.vector.keys())
print("Vector length:", len(obj.vector["default"]))

{'answer': 'Italy', 'category': 'Culture', 'question': 'Leonardo da Vinci was born in this country.'}
Vector keys: dict_keys(['default'])
Vector length: 1536


In [8]:
questions.data.update(
    uuid=uuid,
    properties={
        "answer": "Florence, Italy",
    },
)

obj = questions.query.fetch_object_by_id(uuid=uuid)

print(obj.properties)

{'answer': 'Florence, Italy', 'category': 'Culture', 'question': 'Leonardo da Vinci was born in this country.'}


In [9]:
questions = client.collections.get("Question")

In [10]:
new_uuid = questions.data.insert(
    {
        "question": "This planet is known as the Red Planet.",
        "answer": "Mars",
        "category": "Science",
    }
)

print("Inserted UUID:", new_uuid)

Inserted UUID: 91d8c789-a254-4e3b-9c97-c9e737b5870b


In [11]:
obj = questions.query.fetch_object_by_id(uuid=new_uuid)

print(obj.uuid)
print(obj.properties)

91d8c789-a254-4e3b-9c97-c9e737b5870b
{'answer': 'Mars', 'category': 'Science', 'question': 'This planet is known as the Red Planet.'}


In [12]:
questions.data.update(
    uuid=new_uuid,
    properties={
        "answer": "Mars",
        "category": "Astronomy",
    },
)

In [13]:
updated_obj = questions.query.fetch_object_by_id(uuid=new_uuid)

print(updated_obj.uuid)
print(updated_obj.properties)

91d8c789-a254-4e3b-9c97-c9e737b5870b
{'question': 'This planet is known as the Red Planet.', 'category': 'Astronomy', 'answer': 'Mars'}


In [14]:
response = questions.query.fetch_objects(
    limit=10
)

for obj in response.objects:
    print(obj.uuid)
    print(obj.properties)
    print("-" * 50)

91d8c789-a254-4e3b-9c97-c9e737b5870b
{'question': 'This planet is known as the Red Planet.', 'category': 'Astronomy', 'answer': 'Mars'}
--------------------------------------------------
e84322c0-4614-4180-8911-1d0f855064a3
{'answer': 'Florence, Italy', 'category': 'Culture', 'question': 'Leonardo da Vinci was born in this country.'}
--------------------------------------------------


In [15]:
new_uuid = questions.data.insert(
    {
        "question": "This car is known as the fastest Italian car.",
        "answer": "Ferrari",
        "category": "Cars",
    }
)

print("Inserted UUID:", new_uuid)

Inserted UUID: 6257d17a-e4fe-4145-892c-d2a450fb3d8b


In [16]:
obj = questions.query.fetch_object_by_id(uuid=new_uuid)

print(obj.uuid)
print(obj.properties)

6257d17a-e4fe-4145-892c-d2a450fb3d8b
{'answer': 'Ferrari', 'category': 'Cars', 'question': 'This car is known as the fastest Italian car.'}


In [17]:
questions.data.update(
    uuid=new_uuid,
    properties={
        "answer": "Ferrari",
        "category": "Motor Sport",
    },
)

In [18]:
updated_obj = questions.query.fetch_object_by_id(uuid=new_uuid)

print(updated_obj.uuid)
print(updated_obj.properties)

6257d17a-e4fe-4145-892c-d2a450fb3d8b
{'answer': 'Ferrari', 'category': 'Motor Sport', 'question': 'This car is known as the fastest Italian car.'}


In [19]:
updated_obj = questions.query.fetch_object_by_id(uuid=new_uuid)

print(updated_obj.uuid)
print(updated_obj.properties)

6257d17a-e4fe-4145-892c-d2a450fb3d8b
{'answer': 'Ferrari', 'category': 'Motor Sport', 'question': 'This car is known as the fastest Italian car.'}


In [20]:
response = questions.query.fetch_objects(
    limit=10
)

for obj in response.objects:
    print(obj.uuid)
    print(obj.properties)
    print("-" * 50)

6257d17a-e4fe-4145-892c-d2a450fb3d8b
{'question': 'This car is known as the fastest Italian car.', 'category': 'Motor Sport', 'answer': 'Ferrari'}
--------------------------------------------------
91d8c789-a254-4e3b-9c97-c9e737b5870b
{'answer': 'Mars', 'category': 'Astronomy', 'question': 'This planet is known as the Red Planet.'}
--------------------------------------------------
e84322c0-4614-4180-8911-1d0f855064a3
{'answer': 'Florence, Italy', 'category': 'Culture', 'question': 'Leonardo da Vinci was born in this country.'}
--------------------------------------------------


In [21]:
questions = client.collections.get("Question")

new_uuid = questions.data.insert(
    {
        "question": "This singer is known as king of rock and roll.",
        "answer": "Elvis Presley",
        "category": "Music artists",
    }
)

print("Inserted UUID:", new_uuid)

obj = questions.query.fetch_object_by_id(uuid=new_uuid)
print("Before update:")
print(obj.properties)

questions.data.update(
    uuid=new_uuid,
    properties={
        "answer": "Elvis Presley",
        "category": "Music artists and entertainment",
    },
)

updated_obj = questions.query.fetch_object_by_id(uuid=new_uuid)
print("After update:")
print(updated_obj.properties)

Inserted UUID: f5d4caa8-59e4-459e-9d4f-30a0bbbfea06
Before update:
{'answer': 'Elvis Presley', 'category': 'Music artists', 'question': 'This singer is known as king of rock and roll.'}
After update:
{'answer': 'Elvis Presley', 'category': 'Music artists and entertainment', 'question': 'This singer is known as king of rock and roll.'}


In [22]:
response = questions.query.fetch_objects(limit=20)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Properties:", obj.properties)
    print("-" * 60)

UUID: 6257d17a-e4fe-4145-892c-d2a450fb3d8b
Properties: {'question': 'This car is known as the fastest Italian car.', 'category': 'Motor Sport', 'answer': 'Ferrari'}
------------------------------------------------------------
UUID: 91d8c789-a254-4e3b-9c97-c9e737b5870b
Properties: {'answer': 'Mars', 'category': 'Astronomy', 'question': 'This planet is known as the Red Planet.'}
------------------------------------------------------------
UUID: e84322c0-4614-4180-8911-1d0f855064a3
Properties: {'answer': 'Florence, Italy', 'category': 'Culture', 'question': 'Leonardo da Vinci was born in this country.'}
------------------------------------------------------------
UUID: f5d4caa8-59e4-459e-9d4f-30a0bbbfea06
Properties: {'answer': 'Elvis Presley', 'category': 'Music artists and entertainment', 'question': 'This singer is known as king of rock and roll.'}
------------------------------------------------------------


In [23]:
questions.data.delete_by_id("e84322c0-4614-4180-8911-1d0f855064a3")

True

In [25]:
response = questions.query.fetch_objects(limit=20)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Properties:", obj.properties)
    print("-" * 60)

UUID: 6257d17a-e4fe-4145-892c-d2a450fb3d8b
Properties: {'answer': 'Ferrari', 'category': 'Motor Sport', 'question': 'This car is known as the fastest Italian car.'}
------------------------------------------------------------
UUID: 91d8c789-a254-4e3b-9c97-c9e737b5870b
Properties: {'answer': 'Mars', 'category': 'Astronomy', 'question': 'This planet is known as the Red Planet.'}
------------------------------------------------------------
UUID: f5d4caa8-59e4-459e-9d4f-30a0bbbfea06
Properties: {'answer': 'Elvis Presley', 'category': 'Music artists and entertainment', 'question': 'This singer is known as king of rock and roll.'}
------------------------------------------------------------


In [26]:
import weaviate.classes as wvc

collection_name = "Article"

if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

articles = client.collections.create(
    name=collection_name,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small"
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="author",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Collection created:", collection_name)

Collection created: Article


In [27]:
article_uuid = articles.data.insert(
    {
        "title": "Introduction to Vector Databases",
        "content": "Vector databases store embeddings and allow semantic search.",
        "author": "Piotr",
        "category": "AI",
    }
)

print("Inserted UUID:", article_uuid)

Inserted UUID: 9bfb23b9-67ed-4703-9cb5-2f138356f2cd


In [28]:
article = articles.query.fetch_object_by_id(uuid=article_uuid)

print(article.uuid)
print(article.properties)

9bfb23b9-67ed-4703-9cb5-2f138356f2cd
{'content': 'Vector databases store embeddings and allow semantic search.', 'title': 'Introduction to Vector Databases', 'category': 'AI', 'author': 'Piotr'}


In [29]:
articles.data.update(
    uuid=article_uuid,
    properties={
        "title": "Introduction to Vector Databases and Weaviate",
        "category": "Vector Search",
    },
)

In [30]:
updated_article = articles.query.fetch_object_by_id(uuid=article_uuid)

print(updated_article.uuid)
print(updated_article.properties)

9bfb23b9-67ed-4703-9cb5-2f138356f2cd
{'title': 'Introduction to Vector Databases and Weaviate', 'content': 'Vector databases store embeddings and allow semantic search.', 'category': 'Vector Search', 'author': 'Piotr'}


In [31]:
collections = client.collections.list_all()

for name in collections:
    print(name)

Article
Question


In [32]:
articles = client.collections.get("Article")

article_objects = [
    {
        "title": "Introduction to Vector Databases",
        "content": "Vector databases store embeddings and allow semantic search over unstructured data.",
        "author": "Piotr",
        "category": "AI",
    },
    {
        "title": "What is Weaviate?",
        "content": "Weaviate is a vector database that supports semantic search, hybrid search, and generative search.",
        "author": "Piotr",
        "category": "Vector Database",
    },
    {
        "title": "Semantic Search Basics",
        "content": "Semantic search compares the meaning of text using vector embeddings instead of only matching keywords.",
        "author": "Piotr",
        "category": "Search",
    },
    {
        "title": "Hybrid Search in Practice",
        "content": "Hybrid search combines keyword-based search with vector search to improve search result quality.",
        "author": "Piotr",
        "category": "Search",
    },
    {
        "title": "Embeddings Explained",
        "content": "Embeddings are numeric vector representations of text, images, or other data types.",
        "author": "Piotr",
        "category": "Machine Learning",
    },
]

inserted_uuids = []

for article in article_objects:
    uuid = articles.data.insert(article)
    inserted_uuids.append(uuid)
    print("Inserted:", uuid, article["title"])

print("\nAll inserted UUIDs:")
for uuid in inserted_uuids:
    print(uuid)

Inserted: 548518e8-eb0e-4499-8e32-f21c5cdffeb7 Introduction to Vector Databases
Inserted: f2264ec9-45d7-4042-a540-9d02fd4bd73d What is Weaviate?
Inserted: e052f3c3-2b4e-4ce1-b813-0485d1ffa693 Semantic Search Basics
Inserted: b860ea92-6e37-4905-9e3c-1d4466eeb022 Hybrid Search in Practice
Inserted: 52828122-1488-40a8-b63e-dc41ef693271 Embeddings Explained

All inserted UUIDs:
548518e8-eb0e-4499-8e32-f21c5cdffeb7
f2264ec9-45d7-4042-a540-9d02fd4bd73d
e052f3c3-2b4e-4ce1-b813-0485d1ffa693
b860ea92-6e37-4905-9e3c-1d4466eeb022
52828122-1488-40a8-b63e-dc41ef693271


In [33]:
article_uuid = inserted_uuids[0]

article = articles.query.fetch_object_by_id(uuid=article_uuid)

print(article.uuid)
print(article.properties)

548518e8-eb0e-4499-8e32-f21c5cdffeb7
{'author': 'Piotr', 'title': 'Introduction to Vector Databases', 'category': 'AI', 'content': 'Vector databases store embeddings and allow semantic search over unstructured data.'}


In [34]:
article_uuid = inserted_uuids[0]

articles.data.update(
    uuid=article_uuid,
    properties={
        "title": "Introduction to Vector Databases with Weaviate",
        "category": "Weaviate",
    },
)

updated_article = articles.query.fetch_object_by_id(uuid=article_uuid)

print(updated_article.uuid)
print(updated_article.properties)

548518e8-eb0e-4499-8e32-f21c5cdffeb7
{'content': 'Vector databases store embeddings and allow semantic search over unstructured data.', 'author': 'Piotr', 'category': 'Weaviate', 'title': 'Introduction to Vector Databases with Weaviate'}


In [35]:
client.close()